### Align MOD16 to 500m

In [ ]:
import os
import rasterio
from rasterio.warp import reproject, Resampling

# ======================
# FOLDERS
# ======================
pet_folder = r"C:\Users\gerii\Documents\projects\tekeze\MOD16"
ref_folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS_500m"

out_folder = r"C:\Users\gerii\Documents\projects\tekeze\MOD16_500m"
os.makedirs(out_folder, exist_ok=True)

# ======================
# LOOP FILES
# ======================
files = [f for f in os.listdir(pet_folder) if f.endswith(".tif")]

for f in files:

    pet_path = os.path.join(pet_folder, f)

    # match CHIRPS file
    date = f.replace("PET_", "")
    ref_path = os.path.join(ref_folder, date)

    if not os.path.exists(ref_path):
        continue

    with rasterio.open(ref_path) as ref:
        ref_data = ref.read(1)
        ref_transform = ref.transform
        ref_crs = ref.crs
        ref_shape = ref_data.shape

    with rasterio.open(pet_path) as src:

        pet_data = src.read(1)

        dst = rasterio.open(
            os.path.join(out_folder, f),
            'w',
            driver='GTiff',
            height=ref_shape[0],
            width=ref_shape[1],
            count=1,
            dtype=pet_data.dtype,
            crs=ref_crs,
            transform=ref_transform,
        )

        reproject(
            source=pet_data,
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.bilinear   # ✔ best for PET
        )

        dst.close()

    print("Aligned:", f)

print("All PET aligned ✔")

### SPEI without using spei library

In [ ]:
import os
import numpy as np
import rasterio
from scipy.ndimage import uniform_filter1d

# ======================
# FOLDERS
# ======================
chirps_folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS_500m"
pet_folder    = r"C:\Users\gerii\Documents\projects\tekeze\MOD16_500m"

out_folder = r"C:\Users\gerii\Documents\projects\tekeze\SPEI"
os.makedirs(out_folder, exist_ok=True)

# ======================
# FILE LIST (2000–2025)
# ======================
years = range(2000, 2026)

dates = []
for y in years:
    for m in range(1, 13):
        dates.append(f"{y}_{m:02d}")

# ======================
# STACK DATA
# ======================
stack_P = []
stack_PET = []

print("Loading data...")

for d in dates:

    chirps_path = os.path.join(chirps_folder, f"{d}.tif")
    pet_path    = os.path.join(pet_folder, f"PET_{d}.tif")

    if not (os.path.exists(chirps_path) and os.path.exists(pet_path)):
        continue

    with rasterio.open(chirps_path) as src:
        P = src.read(1).astype("float32")
        profile = src.profile

    with rasterio.open(pet_path) as src:
        PET = src.read(1).astype("float32")

    # mask invalid
    P[P < 0] = np.nan
    PET[PET < 0] = np.nan

    stack_P.append(P)
    stack_PET.append(PET)

stack_P = np.array(stack_P)
stack_PET = np.array(stack_PET)

print("Stack shape:", stack_P.shape)

# ======================
# WATER BALANCE
# ======================
D = stack_P - stack_PET

# ======================
# FUNCTION: SPEI
# ======================
def compute_spei(data, scale):

    # moving average
    data_scaled = uniform_filter1d(data, size=scale, axis=0, mode='nearest')

    # mean & std
    mean = np.nanmean(data_scaled, axis=0)
    std  = np.nanstd(data_scaled, axis=0)

    # mask invalid pixels (all-NaN or zero variance)
    invalid = np.isnan(mean) | (std == 0)

    std[invalid] = np.nan

    spei = (data_scaled - mean) / std

    # force invalid pixels to NaN
    spei[:, invalid] = np.nan

    return spei

# ======================
# COMPUTE SCALES
# ======================
scales = [1, 3, 6]

for scale in scales:

    print(f"\nProcessing SPEI-{scale}...")

    spei = compute_spei(D, scale)

    scale_folder = os.path.join(out_folder, f"SPEI_{scale}")
    os.makedirs(scale_folder, exist_ok=True)

    for i, d in enumerate(dates[:spei.shape[0]]):

        out_path = os.path.join(scale_folder, f"SPEI{scale}_{d}.tif")

        profile.update(dtype=rasterio.float32, count=1)

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(spei[i].astype("float32"), 1)

    print(f"SPEI-{scale} done ✔")

print("\nAll SPEI scales completed ✔")

### visulaize SPEI3 

In [ ]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# ======================
# FOLDER
# ======================
folder = r"C:\Users\gerii\Documents\projects\tekeze\SPEI\SPEI_6"

files = sorted([
    f for f in os.listdir(folder)
    if f.startswith("SPEI6_2020") and f.endswith(".tif")
])

print("Files found:", len(files))

# ======================
# FIGURE
# ======================
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
axes = axes.flatten()

months = ["Jan","Feb","Mar","Apr","May","Jun",
          "Jul","Aug","Sep","Oct","Nov","Dec"]

# SPEI range (standard)
vmin = -3
vmax = 3

# ======================
# PLOT LOOP
# ======================
for i, f in enumerate(files):

    path = os.path.join(folder, f)

    with rasterio.open(path) as src:
        data = src.read(1).astype("float32")

        # mask invalid
        data[np.abs(data) > 10] = np.nan

        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

    ax = axes[i]

    im = ax.imshow(
        data,
        extent=extent,
        origin="upper",
        cmap="RdBu",   # dry=red, wet=blue
        vmin=vmin,
        vmax=vmax
    )

    ax.set_title(months[i], fontsize=10)

    # ======================
    # 1-degree grid
    # ======================
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.tick_params(labelsize=8)

    # ======================
    # AXIS STYLE
    # ======================
    row = i // 4
    col = i % 4

    if col == 0:
        ax.set_ylabel("Lat", fontsize=9)
    else:
        ax.set_yticklabels([])

    if row == 2:
        ax.set_xlabel("Lon", fontsize=9)
    else:
        ax.set_xticklabels([])

# ======================
# REMOVE EMPTY AXES
# ======================
for j in range(len(files), 12):
    axes[j].axis("off")

# ======================
# LAYOUT (tight like your NDVI)
# ======================
plt.subplots_adjust(
    left=0.06,
    right=0.88,
    top=0.92,
    bottom=0.08,
    wspace=0.02,
    hspace=0.05
)

# ======================
# COLORBAR (right side)
# ======================
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("SPEI-6", fontsize=11)

# ======================
# TITLE
# ======================
plt.suptitle("Monthly SPEI-6 (2020) - Tekeze Basin", fontsize=14)

# ======================
# SAVE
# ======================
fig.canvas.draw()
fig.savefig(
    r"C:\Users\gerii\Documents\projects\tekeze\SPEI\SPEI_6\SPEI6_2020_map.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
import os
import numpy as np
import rasterio

# ======================
# FOLDERS
# ======================
chirps_folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS_500m"
pet_folder    = r"C:\Users\gerii\Documents\projects\tekeze\MOD16_500m"

out_folder = r"C:\Users\gerii\Documents\projects\tekeze\SPEI\SPEI_12"
os.makedirs(out_folder, exist_ok=True)

# ======================
# FILE LIST (2000–2025)
# ======================
years = range(2000, 2026)

dates = []
for y in years:
    for m in range(1, 13):
        dates.append(f"{y}_{m:02d}")

# ======================
# STACK DATA
# ======================
stack_P = []
stack_PET = []

print("Loading data...")

valid_dates = []

for d in dates:

    chirps_path = os.path.join(chirps_folder, f"{d}.tif")
    pet_path    = os.path.join(pet_folder, f"PET_{d}.tif")

    if not (os.path.exists(chirps_path) and os.path.exists(pet_path)):
        print(f"Missing data for {d}")
        continue

    with rasterio.open(chirps_path) as src:
        P = src.read(1).astype("float32")
        profile = src.profile

    with rasterio.open(pet_path) as src:
        PET = src.read(1).astype("float32")

    # mask invalid values
    P[P < 0] = np.nan
    PET[PET < 0] = np.nan

    stack_P.append(P)
    stack_PET.append(PET)

    valid_dates.append(d)

stack_P = np.array(stack_P)
stack_PET = np.array(stack_PET)

print("Stack shape:", stack_P.shape)

# ======================
# WATER BALANCE
# ======================
D = stack_P - stack_PET

# ======================
# COMPUTE SPEI-12
# ======================
print("\nComputing SPEI-12...")

scale = 12

# rolling 12-month mean
D_scaled = np.full_like(D, np.nan)

for t in range(scale - 1, D.shape[0]):

    window = D[t - scale + 1:t + 1]

    D_scaled[t] = np.nanmean(window, axis=0)

# climatological mean and std
mean = np.nanmean(D_scaled, axis=0)
std  = np.nanstd(D_scaled, axis=0)

# invalid pixels
invalid = np.isnan(mean) | (std == 0)

std[invalid] = np.nan

# standardization
spei12 = (D_scaled - mean) / std

# force invalid pixels to NaN
spei12[:, invalid] = np.nan

# ======================
# SAVE OUTPUTS
# ======================
print("\nSaving SPEI-12 rasters...")

profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan
)

for i, d in enumerate(valid_dates):

    out_path = os.path.join(out_folder, f"SPEI12_{d}.tif")

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(spei12[i].astype("float32"), 1)

print("\nSPEI-12 completed successfully ✔")

### Barplot

In [ ]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import pandas as pd

# =====================================================
# MAIN FOLDER
# =====================================================
base_folder = r"C:\Users\gerii\Documents\projects\tekeze\indices\SPEI"

scales = [1, 3, 6, 12]

# =====================================================
# FUNCTION TO LOAD SPEI SERIES
# =====================================================
def load_spei_series(folder, scale):

    scale_folder = os.path.join(folder, f"SPEI_{scale}")

    files = sorted([
        f for f in os.listdir(scale_folder)
        if f.endswith(".tif")
    ])

    dates = []
    values = []

    for f in files:

        path = os.path.join(scale_folder, f)

        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)

        # mask invalid values
        arr[arr < -9990] = np.nan

        # spatial mean
        mean_val = np.nanmean(arr)

        # filename example:
        # SPEI1_2000_01.tif

        name = os.path.splitext(f)[0]

        parts = name.split("_")

        year = parts[-2]
        month = parts[-1]

        date = pd.to_datetime(f"{year}-{month}")

        dates.append(date)
        values.append(mean_val)

    return dates, values

# =====================================================
# LOAD ALL SPEI DATA
# =====================================================
all_data = {}

for scale in scales:

    print(f"Loading SPEI-{scale}...")

    d, v = load_spei_series(base_folder, scale)

    all_data[scale] = (d, v)

# =====================================================
# CREATE FIGURE
# =====================================================
fig, axes = plt.subplots(
    4,
    1,
    figsize=(20, 14),
    sharex=True
)

# =====================================================
# PLOT EACH SCALE
# =====================================================
for ax, scale in zip(axes, scales):

    dates, values = all_data[scale]

    # color bars
    colors = [
        "blue" if v >= 0 else "red"
        for v in values
    ]

    ax.bar(
        dates,
        values,
        color=colors,
        width=25
    )

    # zero line
    ax.axhline(
        0,
        color='black',
        linewidth=1
    )

    # subplot title
    ax.set_title(
        f"SPEI-{scale}",
        fontsize=25,    
    )
#fontweight='bold'
    # y-axis label
    ax.set_ylabel(
        "SPEI",
        fontsize=25
    )

    # larger tick labels
    ax.tick_params(
        axis='both',
        labelsize=23
    )

    # grid
    ax.grid(
        True,
        linestyle='--',
        alpha=0.4
    )

# =====================================================
# X LABEL
# =====================================================
plt.xlabel(
    "Year",
    fontsize=25
)

# =====================================================
# LAYOUT
# =====================================================
plt.tight_layout()

# =====================================================
# SAVE FIGURE
# =====================================================
out_path = os.path.join(
    base_folder,
    "SPEI_4panel_barplot.png"
)

plt.savefig(
    out_path,
    dpi=350,
    bbox_inches='tight'
)

print(f"\nSaved figure:\n{out_path}")

# =====================================================
# SHOW FIGURE
# =====================================================
plt.show()

### fitted probability distribution, parameter-estimation method 

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from scipy.special import gamma as G
from scipy.stats import norm, skew, probplot

# ======================
# FOLDERS
# ======================
chirps_folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS_500m"
pet_folder    = r"C:\Users\gerii\Documents\projects\tekeze\MOD16_500m"
out_folder    = r"C:\Users\gerii\Documents\projects\tekeze\SPEI_DIAGNOSTICS"
os.makedirs(out_folder, exist_ok=True)

SCALES = [1, 3, 6, 12]      # timescales summarised in the table
FIG_SCALE = 3               # timescale shown in the figure (the one used in the CDI)
KS_CRIT = 1.36 / np.sqrt(26)  # K-S 5% critical distance for n = 26 (descriptive only)

# ======================
# LOAD P AND PET (calendar-aligned, missing months kept as NaN)
# ======================
dates = [f"{y}_{m:02d}" for y in range(2000, 2026) for m in range(1, 13)]
T = len(dates)

def read_tif(path):
    with rasterio.open(path) as src:
        return src.read(1).astype("float32")

first = next(p for p in (os.path.join(chirps_folder, f"{d}.tif") for d in dates) if os.path.exists(p))
R, C = read_tif(first).shape

P   = np.full((T, R, C), np.nan, dtype="float32")
PET = np.full((T, R, C), np.nan, dtype="float32")
missing_files = []

print("Loading data...")
for i, d in enumerate(dates):
    pp = os.path.join(chirps_folder, f"{d}.tif")
    ep = os.path.join(pet_folder, f"PET_{d}.tif")
    if os.path.exists(pp):
        P[i] = read_tif(pp)
    else:
        missing_files.append(("CHIRPS", d))
    if os.path.exists(ep):
        PET[i] = read_tif(ep)
    else:
        missing_files.append(("PET", d))

P[P < 0] = np.nan
PET[PET < 0] = np.nan
D = P - PET

land = np.isfinite(D).any(axis=0)                 # pixels with any data
n_land = int(land.sum())
miss_P   = 100 * np.mean(~np.isfinite(P[:, land]))
miss_PET = 100 * np.mean(~np.isfinite(PET[:, land]))
miss_D   = 100 * np.mean(~np.isfinite(D[:, land]))

# ======================
# LOG-LOGISTIC (PWM) FIT AND SPEI
# ======================
def pwm_loglogistic(X):
    n = X.shape[0]
    Xs = np.sort(X, axis=0)
    Fi = ((np.arange(1, n + 1) - 0.35) / n).reshape(-1, 1, 1)
    w0 = Xs.mean(axis=0)
    w1 = ((1 - Fi) * Xs).mean(axis=0)
    w2 = (((1 - Fi) ** 2) * Xs).mean(axis=0)
    with np.errstate(all="ignore"):
        beta = (2 * w1 - w0) / (6 * w1 - w0 - 6 * w2)
        g1, g2 = G(1 + 1 / beta), G(1 - 1 / beta)
        alpha = (w0 - 2 * w1) * beta / (g1 * g2)
        gam = w0 - alpha * g1 * g2
    return alpha, beta, gam

def loglogistic_cdf(X, alpha, beta, gam):
    with np.errstate(all="ignore"):
        F = 1.0 / (1.0 + (alpha / (X - gam)) ** beta)
        return np.where(X - gam > 0, F, 0.0)

def fit_spei(D, scale):
    T = D.shape[0]
    acc = np.full_like(D, np.nan)
    for t in range(scale - 1, T):
        acc[t] = D[t - scale + 1:t + 1].sum(axis=0)

    spei = np.full_like(D, np.nan)
    A = np.full((12,) + D.shape[1:], np.nan, dtype="float32")
    B, GM, KS = A.copy(), A.copy(), A.copy()
    n_used = np.zeros(12, dtype=int)
    n_series = n_incomplete = n_invalid = 0

    for m in range(12):
        idx = np.arange(m, T, 12)
        idx = idx[np.isfinite(acc[idx]).any(axis=(1, 2))]
        if len(idx) < 10:
            continue
        X = acc[idx]
        n = X.shape[0]
        n_used[m] = n
        has_data = np.isfinite(X).any(axis=0)
        complete = np.all(np.isfinite(X), axis=0)

        a, b, g = pwm_loglogistic(X)
        F = loglogistic_cdf(X, a, b, g)
        Z = norm.ppf(np.clip(F, 1e-6, 1 - 1e-6))

        valid = complete & np.isfinite(b) & (b > 1)
        Z[:, ~valid] = np.nan
        spei[idx] = Z
        A[m], B[m], GM[m] = np.where(valid, a, np.nan), np.where(valid, b, np.nan), np.where(valid, g, np.nan)

        # Kolmogorov-Smirnov distance between fitted and empirical CDF
        Xs = np.sort(X, axis=0)
        Fe = ((np.arange(1, n + 1) - 0.5) / n).reshape(-1, 1, 1)
        ks = np.abs(loglogistic_cdf(Xs, a, b, g) - Fe).max(axis=0)
        KS[m] = np.where(valid, ks, np.nan)

        n_series     += int(has_data.sum())
        n_incomplete += int((has_data & ~complete).sum())
        n_invalid    += int((complete & ~valid).sum())

    return spei, A, B, GM, KS, n_used, (n_series, n_incomplete, n_invalid)

# ======================
# RUN ALL SCALES
# ======================
rows = []
keep = {}
for k in SCALES:
    print(f"Fitting SPEI-{k} ...")
    spei, A, B, GM, KS, n_used, (n_series, n_inc, n_inv) = fit_spei(D, k)
    z = spei[np.isfinite(spei)]
    mm = [np.nanmean(spei[m::12]) for m in range(12)]
    ss = [np.nanstd(spei[m::12]) for m in range(12)]
    ks = KS[np.isfinite(KS)]
    rows.append({
        "Scale (months)": k,
        "Years per calendar month": int(n_used.max()),
        "Pixel-month series": n_series,
        "Incomplete series (%)": round(100 * n_inc / n_series, 2),
        "Invalid fits, beta<=1 (%)": round(100 * n_inv / n_series, 2),
        "Valid fits (%)": round(100 * (n_series - n_inc - n_inv) / n_series, 2),
        "Median beta": round(float(np.nanmedian(B)), 2),
        "Mean K-S distance": round(float(ks.mean()), 3),
        "K-S below 5% critical (%)": round(100 * float((ks < KS_CRIT).mean()), 1),
        "SPEI mean": round(float(z.mean()), 3),
        "SPEI SD": round(float(z.std()), 3),
        "SPEI skewness": round(float(skew(z)), 3),
        "SPEI at probability limit (%)": round(100 * float((np.abs(z) > 4.7).mean()), 3),
        "Monthly mean range": f"{min(mm):+.2f} to {max(mm):+.2f}",
        "Monthly SD range": f"{min(ss):.2f} to {max(ss):.2f}",
    })
    if k == FIG_SCALE:
        keep = dict(spei=spei, B=B, A=A, GM=GM, KS=KS)

summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(out_folder, "SPEI_fit_summary.csv"), index=False)

# ======================
# METHODS SUMMARY (numbers to quote in the manuscript)
# ======================
print("\n" + "=" * 60)
print("METHODS SUMMARY")
print("=" * 60)
print("Distribution          : three-parameter log-logistic")
print("Estimation            : probability-weighted moments, F_i = (i-0.35)/N")
print(f"Reference period      : {dates[0]} to {dates[-1]} ({T} months, 26 years)")
print(f"Pixels with data      : {n_land}")
print(f"Missing files         : {len(missing_files)} {missing_files[:5]}")
print(f"Missing P / PET / D   : {miss_P:.2f}% / {miss_PET:.2f}% / {miss_D:.2f}% of pixel-months")
print("Transformation        : SPEI = Phi^-1( F(D_k) ), per pixel, calendar month, timescale\n")
print(summary.T.to_string(header=False))

# ======================
# FIGURE
# ======================
plt.rcParams.update({"font.size": 16, "font.family": "serif"})
fig, ax = plt.subplots(2, 2, figsize=(16, 13))

# (a) fitted vs empirical CDF of the k-month water balance at a representative pixel
months4 = [0, 3, 6, 9]
ok4 = np.isfinite(keep["B"][months4]).all(axis=0) & land
ks_pix = np.nanmean(keep["KS"], axis=0)
cand = np.argwhere(ok4)
med_ks = np.median(ks_pix[ok4])
r0, c0 = cand[np.argmin(np.abs(ks_pix[ok4] - med_ks))]      # pixel with median K-S distance
series = D[:, r0, c0]
acc = np.full(len(series), np.nan)
for t in range(FIG_SCALE - 1, len(series)):
    acc[t] = series[t - FIG_SCALE + 1:t + 1].sum()
for m, name, col in [(0, "Jan", "#1b9e77"), (3, "Apr", "#d95f02"), (6, "Jul", "#7570b3"), (9, "Oct", "#e7298a")]:
    x = acc[m::12]
    x = x[np.isfinite(x)]
    n = len(x)
    xs = np.sort(x)
    a, b, g = pwm_loglogistic(xs.reshape(-1, 1, 1))
    xx = np.linspace(xs.min() - 0.1 * np.ptp(xs), xs.max() + 0.1 * np.ptp(xs), 300)
    ax[0, 0].plot(xs, (np.arange(1, n + 1) - 0.35) / n, "o", color=col, ms=6)
    ax[0, 0].plot(xx, loglogistic_cdf(xx, a[0, 0], b[0, 0], g[0, 0]), "-", color=col, lw=2, label=name)
ax[0, 0].set_xlabel(f"{FIG_SCALE}-month water balance (mm)")
ax[0, 0].set_ylabel("Cumulative probability")
ax[0, 0].set_title("(a) Empirical (dots) and fitted CDF (lines)", fontsize=16)
ax[0, 0].legend(); ax[0, 0].grid(True, ls="--", alpha=0.4)

# (b) Q-Q plot of SPEI against the standard normal
z = keep["spei"][np.isfinite(keep["spei"])]
z = np.random.default_rng(0).choice(z, size=min(200000, z.size), replace=False)
(osm, osr), _ = probplot(z, dist="norm")
ax[0, 1].plot(osm, osr, ".", ms=2, alpha=0.4)
lim = [-4.5, 4.5]
ax[0, 1].plot(lim, lim, "k--", lw=2)
ax[0, 1].set_xlim(lim); ax[0, 1].set_ylim(lim)
ax[0, 1].set_xlabel("Standard normal quantiles"); ax[0, 1].set_ylabel(f"SPEI-{FIG_SCALE} quantiles")
ax[0, 1].set_title("(b) Normality of SPEI", fontsize=16)
ax[0, 1].grid(True, ls="--", alpha=0.4)

# (c) mean shape parameter, (d) share of calendar months with a valid fit
beta_mean = np.nanmean(keep["B"], axis=0)
valid_pct = 100 * np.isfinite(keep["B"]).sum(axis=0) / 12.0
valid_pct = np.where(land, valid_pct, np.nan)
for a_, arr, ttl, cm in [(ax[1, 0], beta_mean, "(c) Mean shape parameter $\\beta$ (2-98th pct. scale)", "viridis"),
                         (ax[1, 1], valid_pct, "(d) Months with a valid fit (%)", "YlGn")]:
    lo, hi = np.nanpercentile(arr, [2, 98])
    im = a_.imshow(arr, cmap=cm, vmin=lo, vmax=hi)
    a_.set_title(ttl, fontsize=16); a_.axis("off")
    fig.colorbar(im, ax=a_, fraction=0.04, pad=0.02)

plt.tight_layout()
fig_path = os.path.join(out_folder, "SPEI_diagnostics.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print("\nFigure saved at:", fig_path)
print("Table saved at :", os.path.join(out_folder, "SPEI_fit_summary.csv"))

In [5]:
import os
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from scipy.special import gamma as G

# ======================
# FOLDERS AND SETTINGS
# ======================
chirps_folder = r"C:\Users\gerii\Documents\projects\tekeze\CHIRPS_500m"
pet_folder    = r"C:\Users\gerii\Documents\projects\tekeze\MOD16_500m"
out_folder    = r"C:\Users\gerii\Documents\projects\tekeze\SPEI_DIAGNOSTICS"
os.makedirs(out_folder, exist_ok=True)

SCALES = [1, 3, 6, 12]
MNAME = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
PER_MONTH_FIGS = True       # 12 four-panel PNGs (one per calendar month)

# ======================
# LOAD P AND PET
# ======================
dates = [f"{y}_{m:02d}" for y in range(2000, 2026) for m in range(1, 13)]
T = len(dates)

def read_tif(path):
    with rasterio.open(path) as src:
        return src.read(1).astype("float32")

first = next(p for p in (os.path.join(chirps_folder, f"{d}.tif") for d in dates) if os.path.exists(p))
R, C = read_tif(first).shape
P   = np.full((T, R, C), np.nan, dtype="float32")
PET = np.full((T, R, C), np.nan, dtype="float32")
missing_files = 0
print("Loading data...")
for i, d in enumerate(dates):
    pp = os.path.join(chirps_folder, f"{d}.tif")
    ep = os.path.join(pet_folder, f"PET_{d}.tif")
    if os.path.exists(pp): P[i] = read_tif(pp)
    else: missing_files += 1
    if os.path.exists(ep): PET[i] = read_tif(ep)
    else: missing_files += 1
P[P < 0] = np.nan
PET[PET < 0] = np.nan
D = P - PET
land = np.isfinite(D).any(axis=0)
complete_all = np.all(np.isfinite(D), axis=0) & land
print(f"Pixels with data: {int(land.sum())} | fully complete pixels: {int(complete_all.sum())} | missing files: {missing_files}")
if complete_all.sum() == 0:
    raise SystemExit("No pixel has a complete record.")

# ======================
# LOG-LOGISTIC FUNCTIONS
# ======================
def pwm_loglogistic(X):
    """X: (n,) for one series, or (n,R,C) for rasters. Returns alpha, beta, gamma."""
    n = X.shape[0]
    Xs = np.sort(X, axis=0)
    Fi = ((np.arange(1, n + 1) - 0.35) / n).reshape((-1,) + (1,) * (X.ndim - 1))
    w0 = Xs.mean(axis=0)
    w1 = ((1 - Fi) * Xs).mean(axis=0)
    w2 = (((1 - Fi) ** 2) * Xs).mean(axis=0)
    with np.errstate(all="ignore"):
        beta = (2 * w1 - w0) / (6 * w1 - w0 - 6 * w2)
        g1, g2 = G(1 + 1 / beta), G(1 - 1 / beta)
        alpha = (w0 - 2 * w1) * beta / (g1 * g2)
        gam = w0 - alpha * g1 * g2
    return alpha, beta, gam

def ll_cdf(X, a, b, g):
    with np.errstate(all="ignore"):
        F = 1.0 / (1.0 + (a / (X - g)) ** b)
        return np.where(X - g > 0, F, 0.0)

def ll_pdf(x, a, b, g):
    u = (x - g) / a
    return (b / a) * u ** (b - 1) * (1 + u ** b) ** -2      # only for x > g

def accumulate(D, k):
    acc = np.full_like(D, np.nan)
    for t in range(k - 1, D.shape[0]):
        acc[t] = D[t - k + 1:t + 1].sum(axis=0)
    return acc

def pixel_acc(series, k):
    out = np.full(len(series), np.nan)
    for t in range(k - 1, len(series)):
        out[t] = series[t - k + 1:t + 1].sum()
    return out

# ======================
# FIT ALL SCALES AND MONTHS
# ======================
overall, permonth = [], []
count_valid = np.zeros((R, C), dtype=np.int16)     # number of valid (scale, month) fits per pixel
ks_sum = np.zeros((R, C), dtype="float32")
for k in SCALES:
    print(f"Fitting SPEI-{k} ...")
    acc = accumulate(D, k)
    n_series = n_inc = n_inv = 0
    A_all, B_all, G_all, K_all = [], [], [], []
    for m in range(12):
        idx = np.arange(m, T, 12)
        idx = idx[np.isfinite(acc[idx]).any(axis=(1, 2))]
        if len(idx) < 10:
            continue
        X = acc[idx]
        n = X.shape[0]
        has = np.isfinite(X).any(axis=0)
        comp = np.all(np.isfinite(X), axis=0)
        Xf = np.where(comp[None], X, 0.0)
        a, b, g = pwm_loglogistic(Xf)
        valid = comp & np.isfinite(a) & np.isfinite(b) & np.isfinite(g) & (b > 1)
        Xs = np.sort(Xf, axis=0)
        Fe = ((np.arange(1, n + 1) - 0.5) / n).reshape(-1, 1, 1)
        ks = np.abs(ll_cdf(Xs, a, b, g) - Fe).max(axis=0)
        valid &= np.isfinite(ks)

        count_valid += valid
        ks_sum += np.where(valid, ks, 0.0).astype("float32")
        n_series += int(has.sum()); n_inc += int((has & ~comp).sum()); n_inv += int((comp & ~valid).sum())
        if valid.sum() > 0:
            A_all.append(a[valid]); B_all.append(b[valid]); G_all.append(g[valid]); K_all.append(ks[valid])
            permonth.append({"Scale": k, "Month": MNAME[m], "Years": n, "Valid pixels": int(valid.sum()),
                             "alpha": round(float(np.median(a[valid])), 2),
                             "beta": round(float(np.median(b[valid])), 3),
                             "gamma": round(float(np.median(g[valid])), 2),
                             "KS": round(float(np.mean(ks[valid])), 3)})
    if not A_all:
        raise SystemExit(f"No valid log-logistic fit for SPEI-{k}.")
    A_all, B_all, G_all, K_all = map(np.concatenate, (A_all, B_all, G_all, K_all))
    overall.append({"Scale": k, "Pixel-month series": n_series,
                    "Incomplete (%)": round(100 * n_inc / n_series, 2),
                    "Invalid fit, beta<=1 (%)": round(100 * n_inv / n_series, 2),
                    "Valid fits (%)": round(100 * (n_series - n_inc - n_inv) / n_series, 2),
                    "Median alpha": round(float(np.median(A_all)), 2),
                    "Median beta": round(float(np.median(B_all)), 3),
                    "Median gamma": round(float(np.median(G_all)), 2),
                    "Mean K-S": round(float(K_all.mean()), 3)})
    del acc

tab_overall = pd.DataFrame(overall)
tab_month = pd.DataFrame(permonth)
tab_avg = (tab_month.groupby("Scale")[["alpha", "beta", "gamma", "KS"]].mean().round(3)
           .rename(columns=lambda c: c + " (mean of monthly medians)").reset_index())
tab_overall.to_csv(os.path.join(out_folder, "SPEI_params_overall.csv"), index=False)
tab_month.to_csv(os.path.join(out_folder, "SPEI_params_by_month.csv"), index=False)
tab_avg.to_csv(os.path.join(out_folder, "SPEI_params_monthly_average.csv"), index=False)
print("\nREFERENCE PERIOD: 2000-2025 (26 years); 3-parameter log-logistic; PWM, F_i=(i-0.35)/N")
print(tab_overall.to_string(index=False))
print("\nAverage over the 12 calendar months (of monthly median parameters):")
print(tab_avg.to_string(index=False))

# ======================
# REPRESENTATIVE PIXEL: most valid fits out of 48, then lowest mean K-S
# ======================
best = count_valid[complete_all].max()
cand = complete_all & (count_valid == best)
score = np.where(cand, ks_sum / max(int(best), 1), np.inf)
r0, c0 = np.unravel_index(np.argmin(score), score.shape)
series = D[:, r0, c0].astype("float64")
print(f"\nRepresentative pixel: row={r0}, col={c0} | valid fits: {int(best)} of {12 * len(SCALES)}")

def fit_pixel(k, month):
    x = pixel_acc(series, k)[month - 1::12]
    x = x[np.isfinite(x)]
    a, b, g = [float(v) for v in pwm_loglogistic(x)]
    ok = np.isfinite([a, b, g]).all() and b > 1
    return x, a, b, g, ok

def xgrid(x, a, g):
    xs = np.sort(x); span = np.ptp(xs)
    lo = max(g + 1e-6 * abs(a), xs.min() - 0.25 * span)
    return np.linspace(lo, xs.max() + 0.25 * span, 400)

#plt.rcParams.update({"font.size": 12, "font.family": "serif"})
plt.rcParams.update({"font.size": 22, "font.family": "serif"})
# ======================
# FIGURE SET 1: one 2x2 PNG per calendar month (SPEI-1, 3, 6, 12)
# ======================
if PER_MONTH_FIGS:
    for month in range(1, 13):
        fig, axs = plt.subplots(2, 2, figsize=(13, 9))
        for k, lab, ax in zip(SCALES, "abcd", axs.ravel()):
            x, a, b, g, ok = fit_pixel(k, month)
            ax.hist(x, bins=7, density=True, color="lightgray", edgecolor="k")
            if ok:
                ax.plot(xgrid(x, a, g), ll_pdf(xgrid(x, a, g), a, b, g), "k-", lw=2, label="Log-logistic")
                ax.legend(loc="center right")
                txt = f"$\\alpha$={a:.1f}\n$\\beta$={b:.2f}\n$\\gamma$={g:.1f}\nn={len(x)}"
            else:
                txt = f"$\\beta$={b:.2f} (<=1)\nPDF not defined\nn={len(x)}"
            ax.text(0.97, 0.95, txt, transform=ax.transAxes, ha="right", va="top", fontsize=11,
                    bbox=dict(fc="white", ec="gray"))
            ax.set_title(f"({lab}) SPEI-{k}", fontweight="bold")
            ax.set_xlabel("P - PET (mm)"); ax.set_ylabel("f(x)"); ax.grid(True, ls="--", alpha=0.4)
        fig.suptitle(f"Fitted log-logistic PDF of the D series ({MNAME[month-1]}, 2000-2025)", y=1.0)
        fig.tight_layout()
        p = os.path.join(out_folder, f"SPEI_PDF_1_3_6_12_{month:02d}_{MNAME[month-1]}.png")
        fig.savefig(p, dpi=300, bbox_inches="tight"); plt.close(fig)
        print("Saved:", p)

# ======================
# FIGURE SET 2: all 12 months overlaid, one panel per scale
# ======================
cols = plt.cm.tab20(np.arange(12))
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
for k, lab, ax in zip(SCALES, "abcd", axs.ravel()):
    for month in range(1, 13):
        x, a, b, g, ok = fit_pixel(k, month)
        if ok:
            xx = xgrid(x, a, g)
            ax.plot(xx, ll_pdf(xx, a, b, g), color=cols[month - 1], lw=2, label=MNAME[month - 1])
    ax.set_title(f"({lab}) SPEI-{k}", fontweight="bold")
    ax.set_xlabel("P - PET (mm)"); ax.set_ylabel("f(x)"); ax.grid(True, ls="--", alpha=0.4)
axs[0, 0].legend(ncol=3, fontsize=9, loc="upper left")
fig.suptitle("Fitted log-logistic PDFs of the D series, all calendar months (2000-2025)", y=1.0)
fig.tight_layout()
p = os.path.join(out_folder, "SPEI_PDF_1_3_6_12_all_months_overlay.png")
fig.savefig(p, dpi=300, bbox_inches="tight"); plt.close(fig)
print("Saved:", p)

# ======================
# FIGURE SET 3: for each scale, a 3x4 grid with all 12 months
# ======================
for k in SCALES:
    fig, axs = plt.subplots(3, 4, figsize=(18, 12))
    for month, ax in zip(range(1, 13), axs.ravel()):
        x, a, b, g, ok = fit_pixel(k, month)
        ax.hist(x, bins=7, density=True, color="lightgray", edgecolor="k")
        if ok:
            xx = xgrid(x, a, g)
            ax.plot(xx, ll_pdf(xx, a, b, g), "k-", lw=2)
            ax.text(0.97, 0.95, f"$\\beta$={b:.2f}", transform=ax.transAxes, ha="right", va="top", fontsize=20)
        else:
            ax.text(0.5, 0.5, f"$\\beta$={b:.2f} (<=1)\nPDF not defined", transform=ax.transAxes,
                    ha="center", va="center", fontsize=20)
        ax.set_title(MNAME[month - 1], fontweight="bold")
        ax.set_xlabel("P - PET (mm)"); ax.set_ylabel("f(x)"); ax.grid(True, ls="--", alpha=0.4)
    fig.suptitle(f"Fitted log-logistic PDF of the D series, SPEI-{k}, all months (2000-2025)", y=1.0)
    fig.tight_layout()
    p = os.path.join(out_folder, f"SPEI_PDF_scale{k}_all_months.png")
    fig.savefig(p, dpi=400, bbox_inches="tight"); plt.close(fig)
    print("Saved:", p)

print("\nPer-month parameters:")
print(tab_month.to_string(index=False))

Loading data...
Pixels with data: 336348 | fully complete pixels: 336348 | missing files: 0
Fitting SPEI-1 ...
Fitting SPEI-3 ...
Fitting SPEI-6 ...
Fitting SPEI-12 ...

REFERENCE PERIOD: 2000-2025 (26 years); 3-parameter log-logistic; PWM, F_i=(i-0.35)/N
 Scale  Pixel-month series  Incomplete (%)  Invalid fit, beta<=1 (%)  Valid fits (%)  Median alpha  Median beta  Median gamma  Mean K-S
     1             4036176             0.0                     32.39           67.61         97.88        7.260       -277.58     0.169
     3             4036176             0.0                     35.58           64.42        152.95        7.256       -753.13     0.180
     6             4036176             0.0                     45.77           54.23        245.47        8.894      -1453.35     0.191
    12             4036176             0.0                     63.35           36.65        520.85        9.333      -2246.76     0.152

Average over the 12 calendar months (of monthly median paramete